In [1]:
#import run_model_preT
#import os

#os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

from adage import Adage as ad
from adage import SeqAdage
#import keras
#import run_model
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
#from imp import reload
#from importlib import reload
#import Adage as ad
from scipy.stats import hypergeom
import csv
import time
import random

#import LinkedAE
#import SeqAdage

2026-05-10 23:48:57.304004: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-10 23:48:57.317423: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778471337.329315 2681820 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778471337.332828 2681820 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778471337.342784 2681820 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
import tensorflow as tf

print("Num CPUs Available: ", len(tf.config.list_physical_devices('CPU')))
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.test.is_built_with_cuda())

In [ ]:
#adage_comp = pandas.read_csv('data_files/ADAGE_compendium.csv')
all_comp = pd.read_csv('../data_files/ecmg_lcn01.csv', index_col = 0) # ,usecols = range(100)
gene_num = np.size(all_comp, 0)
samp_num = np.size(all_comp, 1)
print(gene_num, samp_num)
fig = sns.histplot(pd.melt(all_comp), bins = 50)

In [ ]:
#adage_comp = pandas.read_csv('data_files/ADAGE_compendium.csv')
all_comp = pd.read_csv('../data_files/ecmg_mgs_lcn01.csv', index_col = 0) # ,usecols = range(100)
gene_num = np.size(all_comp, 0)
samp_num = np.size(all_comp, 1)
print(gene_num, samp_num)
fig = sns.histplot(pd.melt(all_comp), bins = 50)

In [ ]:
#adage_comp = pandas.read_csv('data_files/ADAGE_compendium.csv')
all_comp = pd.read_csv('../data_files/ecmgp_lcn01.csv', index_col = 0) # ,usecols = range(100)
gene_num = np.size(all_comp, 0)
samp_num = np.size(all_comp, 1)
print(gene_num, samp_num)
fig = sns.histplot(pd.melt(all_comp), bins = 50)

In [ ]:
#adage_comp = pandas.read_csv('data_files/ADAGE_compendium.csv')
all_comp = pd.read_csv('../data_files/ecmgp_lcn01.csv', index_col = 0) # ,usecols = range(100)
gene_num = np.size(all_comp, 0)
samp_num = np.size(all_comp, 1)
print(gene_num, samp_num)
fig = sns.histplot(pd.melt(all_comp), bins = 50)

In [ ]:
comps = ["ecmg_mgs_lcn01.csv","ecmgp_lcn01.csv"]


model_dict_post = {}

stime = time.time()
ltime = 0
c = 0



seeds = list(range(0,3))
#seeds = random.sample(range(10000), 10)

for comp in comps:
    for seed in seeds:
        name = 'ad_' + comp + '_2025_09_01_' + str(seed+100)
        print(name)
        ttime = time.time()
        sa = SeqAdage.SeqAdage('../data_files/ecmg_mgs_lcn01.csv',
                                                  seed=seed+100,
                                                  enc_dim = 450,
                                                  kl1=0,
                                                  kl2=0,
                                                  act = "tanh",
                                                  act2="tanh",
                                                  tied = True,
                                                  epochs=10,
                                                  init="glorot_uniform",
                                                  batch_size=15,
                                                  dropout = 0,
                                                  mm = 0.6,
                                                  lr = 0.5) # run_model.run_model
        #mseq = sa.run_model()
        mseq = sa.train_model()
        temp = ad.Adage(sa.autoencoder, sa.history, sa.all_comp)
       
        model_dict_post[name] = temp
        ltime = ((time.time() - ttime) + ltime)
        c+=1

rtime = time.time() - stime

ad_ecmg_mgs_lcn01.csv_2025_09_01_100


In [ ]:
print(rtime)
print(c)
print(ltime / c)
print(rtime / 60)

In [ ]:
len(seeds)

In [ ]:
name = 'ad_' + comp + '_2025_09_01_' + str(seed+100)
print(name)
model_temp = model_dict_post[name]
print(model_temp.loss)

In [ ]:
model_dict = model_dict_post

xd = len(comps) 
yd = len(seeds)
fig, ax = plt.subplots(xd, yd,figsize=(xd*10 ,yd *4))
fig.tight_layout(pad=3.0)

xi = 0
yi = 0


for comp in comps:
    xi=0
    for seed in seeds:
        name = 'ad_' + comp + '_2025_09_01_' + str(seed+100)
        model_temp = model_dict_post[name]
        ax[yi,xi].plot(list(range(0,100)), model_temp.loss, linewidth=1, markersize=2, color = 'orange')
        ax[yi,xi].plot(list(range(0,100)), model_temp.val_loss, linewidth=1, markersize=2, color = 'red')
        ax[yi,xi].set(title = name, xlabel = 'Epochs', ylabel = 'Loss')
        xi+=1
    yi+=1
        


    

In [ ]:
inits = ["tanh"]
xd = len(comps) 
yd = len(seeds)
fig, ax = plt.subplots(xd, yd,figsize=(yd*8 ,xd *4))
fig.tight_layout(pad=3.0)

xi = 0
yi = 0


for seed in seeds:
    xi=0
    for comp in comps:
        name = 'ad_' + comp + '_2025_09_01_' + str(seed+100)
        model_temp = model_dict_post[name]
        for node in range(0,100):
            sns.histplot(model_temp.weights[node], #[model_temp.weights[node] > 0],
                ax=ax[xi,yi],
                #hist=True,
                binwidth = 0.005, 
                #stat="density",
                kde=False
                #rug=False
                                )
        ax[xi,yi].axvline(x = np.std(model_temp.weights)*2.5)
        ax[xi,yi].axvline(x = np.std(model_temp.weights)*-2.5)
        ax[xi,yi].set(title = name)
        xi+=1
    yi+=1
